# TorchRef Quickstart

From an MTZ file and a starting PDB to a refined structure, refined MTZ, and a 2Fo-Fc map in under 30 lines of code.

If you are running this in Google Colab, run the install cell below. Otherwise skip it.

In [ ]:
# Colab setup - skip if running locally with torchref installed
# !pip install torchref
# !wget -q https://raw.githubusercontent.com/HatPdotS/TorchRef/main/example_notebooks/1DAW.pdb
# !wget -q https://raw.githubusercontent.com/HatPdotS/TorchRef/main/example_notebooks/1DAW.mtz

## 1. Set up paths

1DAW is a small protein (3051 atoms, C2, 2.05 Å). It is shipped with TorchRef next to this notebook and is also downloadable from the repository for Colab.

In [ ]:
import os
import torch
from torchref import ROOT_TORCHREF

if os.path.exists('./1DAW.mtz'):
    mtz_file = './1DAW.mtz'
    pdb_file = './1DAW.pdb'
else:
    mtz_file = f'{ROOT_TORCHREF}/example_notebooks/1DAW.mtz'
    pdb_file = f'{ROOT_TORCHREF}/example_notebooks/1DAW.pdb'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if device != torch.device('cuda'):
    device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
print('device:', device)

## 2. Run a refinement

`LBFGSRefinement` is the recommended entry point. Passing `data_file`, `pdb` and `device` constructs the full pipeline on the requested device:

- reads observations (intensities are converted to amplitudes with French-Wilson),
- loads the atomic model,
- builds geometry restraints from TorchRef's monomer library,
- initialises a `Scaler` with bulk solvent correction,
- registers the standard X-ray, geometry and ADP targets.

`refine(macro_cycles=N)` then alternates scaler / coordinate / ADP optimisation for `N` cycles. Construct on the target device from the start -- moving an already-initialised refinement to GPU mid-run is supported in principle but stresses every internal buffer, so it is safer to pick the device at construction time.

In [ ]:
from torchref import LBFGSRefinement

refinement = LBFGSRefinement(data_file=mtz_file, pdb=pdb_file, device=device)

rwork0, rfree0 = refinement.get_rfactor()
print(f'Initial: Rwork={rwork0:.4f}  Rfree={rfree0:.4f}')

refinement.refine(macro_cycles=3)

rwork, rfree = refinement.get_rfactor()
print(f'Final  : Rwork={rwork:.4f}  Rfree={rfree:.4f}')

## 3. Write refined files

The output MTZ contains the observed amplitudes alongside model amplitudes / phases and the `2mFo-DFc` / `mFo-DFc` map coefficients (column names `FWT/PHWT` and `DELFWT/PHDELWT`), so it can be opened directly in Coot.

In [ ]:
refinement.write_out_pdb('refined.pdb')
refinement.write_out_mtz('refined.mtz')

## 4. Write a CCP4 map

`Map` computes `2Fo-Fc` (default) or `Fcalc` maps on a grid sized from the cell and resolution. `map.write(...)` calculates the map on demand and writes a CCP4 file ready for viewing in PyMOL, Coot or ChimeraX.

In [ ]:
from torchref import Map

two_fo_fc = Map(data=refinement.reflection_data, model=refinement.model, map_type='2Fo-Fc')
two_fo_fc.write('refined_2Fo-Fc.ccp4')

## 5. Controlling which parameters refine

TorchRef exposes parameter control at two levels.

**By parameter type.** `model.freeze(group)` / `unfreeze(group)` operates on whole categories: `'xyz'`, `'adp'` (isotropic B), `'u'` (anisotropic U tensor), `'occupancy'`, `'b'` (alias of `adp`), or `'all'`. `refine_xyz()` and `refine_adp()` are convenience wrappers that freeze the complementary types for one LBFGS pass.

**By atom selection.** `freeze_selection(...)` / `unfreeze_selection(...)` take phenix-style strings (e.g. `'chain A and resseq 10:30'`) and toggle each per-atom mask. For arbitrary per-atom masks you can also write directly to `model.xyz.update_refinable_mask(bool_tensor)`.

In [ ]:
# Selection-based: refine only a 20-residue stretch in chain A
refinement.model.freeze_all()
refinement.model.unfreeze_selection('chain A and resseq 10:30')
refinement.refine_xyz()
refinement.model.unfreeze_all()

# Parameter-type: refine ADPs only, keep coordinates and occupancies fixed
refinement.refine_adp()

# Restore for any further work
refinement.model.unfreeze_all()

## What's next

- `structure_factors.ipynb` -- four ways to compute `F_calc`, from one-liner to manual voxel pipeline; standalone scaling.
- `targets_and_weighting.ipynb` -- explore the standard targets, compare X-ray target modes (Bhattacharyya / ML / LS / Gaussian), see how target-offset weighting controls overfitting, and learn how to write custom targets.
- The CLI tools (`torchref.refine`, `torchref.difference-refine`, `torchref.mtz2map`, ...) wrap exactly the workflow above for headless use.